# Weather API Data Collection

**Goal:** collect addresses and live weather, merge the tables, and save CSV, Excel and SQLite outputs.

**Run order:** run code cells from top to bottom. Keys stay in Colab Secrets. No key should be pasted into a cell.

The location collection covers Germany, Greece, Norway and Iran, followed by five Iranian cities. Missing rain measurements remain blank.

Google Drive is optional. Included coordinates are used when no input CSV is available. Download the final ZIP to keep all results after the runtime disconnects.

## 1. Import libraries


In [ ]:
import pandas as pd
import requests
import json
import time
import sqlite3
from pandas import json_normalize
from pathlib import Path
from io import StringIO
from datetime import datetime, timezone
import importlib.util
import subprocess
import sys
import shutil
import zipfile

## 2. Choose where to save results

Use Drive if it is already mounted. Otherwise use the private Colab runtime and download the ZIP at the end. This avoids a blocked Drive sign-in popup.

In [ ]:
drive_folder = Path('/content/drive/MyDrive/Weather API')
output_base = drive_folder if drive_folder.is_dir() else Path('/content')
run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f_UTC')
output_dir = output_base / 'weather_data_outputs' / run_id
output_dir.mkdir(parents=True, exist_ok=False)
print('Output folder:', output_dir)
print('Drive is connected.' if drive_folder.is_dir() else 'Using Colab storage. Download the final ZIP to keep your results.')

## 3. Load API keys from Colab Secrets


In [ ]:
from google.colab import userdata

locationiq_key = userdata.get('LOCATIONIQ_API_KEY')
openweather_key = userdata.get('OPENWEATHER_API_KEY')
if not locationiq_key or not openweather_key:
    raise RuntimeError('Add both API keys to Colab Secrets and enable notebook access.')
print('Both API secrets are available. Values are hidden.')

## 4. Read the location CSV

The notebook includes 11 sample landmark coordinates. An existing CSV in the configured Drive folder is used first. Weather values are collected from the API.

In [ ]:
file_path = Path('/content/drive/MyDrive/Weather API/world_locations.csv')
coordinate_csv = '\ufeffCountry,Lon,Lat\nUK,-0.1215,51.4995\nUK,-0.1548,51.5229\nUK,-0.1763,51.4959\nUK,-0.1898,50.7477\nUK,-0.1263,51.5187\nFrance,2.2945,48.8584\nFrance,2.3499,48.853\nFrance,2.3376,48.8606\nFrance,2.1204,48.8049\nUSA,-74.0445,40.6892\nUSA,-73.9908,40.7075\n'
world_locations = pd.read_csv(file_path) if file_path.is_file() else pd.read_csv(StringIO(coordinate_csv))
world_locations = world_locations[['Country', 'Lon', 'Lat']].copy()
world_locations[['Lon', 'Lat']] = world_locations[['Lon', 'Lat']].apply(pd.to_numeric, errors='raise')
if world_locations[['Lon', 'Lat']].isna().any().any():
    raise ValueError('Input coordinates must not be missing.')
if world_locations.duplicated(['Lon', 'Lat']).any():
    raise ValueError('Remove duplicate coordinate pairs from the input CSV.')
world_locations.to_csv(output_dir / 'world_locations.csv', index=False)
print('Source:', 'Google Drive CSV' if file_path.is_file() else 'Included coordinate sample')
print('Locations:', len(world_locations))
display(world_locations.head())

## 5. Inspect the first location


In [ ]:
first_location = world_locations.iloc[0]

lat = first_location["Lat"]
lon = first_location["Lon"]

print("Latitude:", lat)
print("Longitude:", lon)

## 6. Define the address function


In [ ]:
# Request handling keeps API keys out of error messages and notebook output.
_api_cache = globals().get('_api_cache', {})

def api_json(endpoint, params, label, cache_key, max_age=600):
    cached = _api_cache.get(cache_key)
    if cached and time.time() - cached[0] < max_age:
        return cached[1]
    for attempt in range(3):
        try:
            response = requests.get(endpoint, params=params, timeout=30)
        except requests.RequestException:
            if attempt == 2:
                raise RuntimeError(f'{label}: connection failed. Check the network and run this cell again.') from None
            time.sleep(2 ** attempt)
            continue
        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
            time.sleep(2 ** (attempt + 1))
            continue
        if response.status_code != 200:
            raise RuntimeError(f'{label}: HTTP {response.status_code}. Check the account, key activation and quota.') from None
        try:
            data = response.json()
        except ValueError:
            raise RuntimeError(f'{label}: the server did not return JSON.') from None
        if isinstance(data, dict) and data.get('error'):
            raise RuntimeError(f'{label}: the provider could not resolve this location.')
        _api_cache[cache_key] = (time.time(), data)
        time.sleep(0.6)
        return data
    raise RuntimeError(f'{label}: request failed after three attempts.')

def get_address_from_lon_lat(entry):
    lat, lon = float(entry['Lat']), float(entry['Lon'])
    return api_json(
        'https://eu1.locationiq.com/v1/reverse',
        {'key': locationiq_key, 'lat': lat, 'lon': lon, 'format': 'json', 'accept-language': 'en'},
        'LocationIQ reverse lookup', ('address', lat, lon), max_age=172800
    )

## 7. Build the address table


In [ ]:
address_df = pd.DataFrame()
address_df['reply'] = world_locations.apply(get_address_from_lon_lat, axis=1)
address_df = json_normalize(address_df['reply'])
address_df.head()

## 8. Select address columns

Some locations have no postcode. Those values stay blank. Coordinates are copied from the input so that the merge uses matching values.

In [ ]:
address_df = address_df.reindex(columns=['address.postcode', 'display_name']).copy()
address_df['Lat'] = world_locations['Lat'].to_numpy()
address_df['Lon'] = world_locations['Lon'].to_numpy()
display(address_df.head())

## 9. Collect live weather

Metric units mean degrees Celsius for temperature and metres per second for wind. The API returns current observations, so values will change between runs.

In [ ]:
def get_weather_city(entry):
    lat, lon = float(entry['Lat']), float(entry['Lon'])
    return api_json(
        'https://api.openweathermap.org/data/2.5/weather',
        {'appid': openweather_key, 'lat': lat, 'lon': lon, 'units': 'metric'},
        'OpenWeather current weather', ('weather', lat, lon)
    )

weather_df = pd.DataFrame()
weather_df['reply'] = world_locations.apply(get_weather_city, axis=1)
weather_df = json_normalize(weather_df['reply'])
display(weather_df.head())

## 10. Select weather columns

An absent rain field means no rain value was supplied. It is kept as missing rather than replaced with an invented measurement.

In [ ]:
weather_columns = ['name', 'main.temp', 'main.temp_min', 'main.temp_max',
                   'main.humidity', 'wind.speed', 'rain.1h']
weather_df = weather_df.reindex(columns=weather_columns).copy()
weather_df['Lat'] = world_locations['Lat'].to_numpy()
weather_df['Lon'] = world_locations['Lon'].to_numpy()
display(weather_df.head())

## 11. Merge and rename

The `County` column contains the locality name returned by OpenWeather; it is not a verified administrative county.

In [ ]:
column_names = {
    'address.postcode': 'Postcode', 'display_name': 'Address',
    'Lat': 'Latitude', 'Lon': 'Longitude', 'name': 'County',
    'main.temp': 'Temperature', 'main.temp_min': 'Temperature_MIN',
    'main.temp_max': 'Temperature_MAX', 'main.humidity': 'Humidity',
    'wind.speed': 'Wind', 'rain.1h': 'Rain_Last_HOUR'
}
Dataset = pd.merge(address_df, weather_df, on=['Lon', 'Lat'], validate='one_to_one')
Dataset = Dataset.rename(columns=column_names)
dataset_columns = list(column_names.values())
Dataset = Dataset.reindex(columns=dataset_columns)
assert len(Dataset) == len(world_locations)
display(Dataset)

## 12. Write and read SQLite

Each full run has a new output folder. Re-running this cell replaces only this run's results table, so it cannot append duplicate rows.

In [ ]:
db_path = output_dir / 'Weather_Database.db'
with sqlite3.connect(db_path) as connection:
    Dataset.to_sql('attractions_weather', connection, if_exists='replace', index=False,
                   dtype={'Postcode': 'TEXT', 'Address': 'TEXT', 'Latitude': 'REAL',
                          'Longitude': 'REAL', 'County': 'TEXT', 'Temperature': 'REAL',
                          'Temperature_MIN': 'REAL', 'Temperature_MAX': 'REAL',
                          'Humidity': 'INTEGER', 'Wind': 'REAL', 'Rain_Last_HOUR': 'REAL'})
    saved_attractions = pd.read_sql_query('SELECT * FROM attractions_weather', connection)
assert len(saved_attractions) == len(Dataset)
display(saved_attractions.head())
print('SQLite read-back passed:', len(saved_attractions), 'rows')

## 13. Extend the attraction collection

Add one attraction in Germany, Greece, Norway and Iran. LocationIQ resolves the coordinates instead of guessing them. The combined collection includes locations in Europe, the USA and Iran.

In [ ]:
def resolve_location(name, country, country_code):
    matches = api_json(
        'https://eu1.locationiq.com/v1/search',
        {'key': locationiq_key, 'q': name, 'countrycodes': country_code,
         'format': 'json', 'addressdetails': 1, 'accept-language': 'en', 'limit': 1},
        'LocationIQ place search', ('search', name, country_code), max_age=172800
    )
    if not isinstance(matches, list) or not matches:
        raise RuntimeError(f'No location found for {name}.')
    match = matches[0]
    if match.get('address', {}).get('country_code', '').lower() != country_code:
        raise RuntimeError(f'The country check failed for {name}.')
    return {'Place': name, 'Country': country, 'Lon': float(match['lon']),
            'Lat': float(match['lat']), 'Matched_address': match['display_name']}

def build_dataset(locations):
    locations = locations.reset_index(drop=True)
    if locations.duplicated(['Lon', 'Lat']).any():
        raise ValueError('Each location must have a unique coordinate pair.')
    addresses = json_normalize(locations.apply(get_address_from_lon_lat, axis=1))
    addresses = addresses.reindex(columns=['address.postcode', 'display_name']).copy()
    weather = json_normalize(locations.apply(get_weather_city, axis=1))
    weather = weather.reindex(columns=weather_columns).copy()
    for frame in (addresses, weather):
        frame['Lat'] = locations['Lat'].to_numpy()
        frame['Lon'] = locations['Lon'].to_numpy()
    result = pd.merge(addresses, weather, on=['Lon', 'Lat'], validate='one_to_one')
    return result.rename(columns=column_names).reindex(columns=dataset_columns)

attraction_choices = [
    ('Brandenburg Gate, Berlin', 'Germany', 'de'),
    ('Acropolis, Athens', 'Greece', 'gr'),
    ('Oslo Opera House', 'Norway', 'no'),
    ('Persepolis', 'Iran', 'ir')
]
extra_locations = pd.DataFrame([resolve_location(*choice) for choice in attraction_choices])
extra_dataset = build_dataset(extra_locations)
Attraction_EU = pd.concat([Dataset, extra_dataset], ignore_index=True)
assert len(Attraction_EU) == len(Dataset) + 4
assert not Attraction_EU.duplicated(['Longitude', 'Latitude']).any()
extra_locations.to_csv(output_dir / 'added_attraction_locations.csv', index=False)
with sqlite3.connect(db_path) as connection:
    Attraction_EU.to_sql('attractions_weather', connection, if_exists='replace', index=False)
display(extra_locations[['Place', 'Country', 'Lat', 'Lon']])
display(Attraction_EU.tail(4))

## 14. Five Iranian cities

Selected country: Iran. Cities: Tehran, Isfahan, Shiraz, Tabriz and Mashhad. Save their results in a separate SQLite table named `Iran`.

In [ ]:
iran_cities = ['Tehran', 'Isfahan', 'Shiraz', 'Tabriz', 'Mashhad']
iran_locations = pd.DataFrame([
    resolve_location(city, 'Iran', 'ir') for city in iran_cities
])
Iran = build_dataset(iran_locations)
Iran.insert(0, 'City', iran_cities)
assert len(Iran) == 5 and Iran['City'].nunique() == 5
iran_locations.to_csv(output_dir / 'Iran_city_locations.csv', index=False)
with sqlite3.connect(db_path) as connection:
    Iran.to_sql('Iran', connection, if_exists='replace', index=False)
    saved_iran = pd.read_sql_query('SELECT * FROM Iran', connection)
assert saved_iran['City'].tolist() == iran_cities
display(saved_iran)

## 15. Export CSV and Excel

Save a genuine Excel 97-2003 `.xls` file and a modern `.xlsx` file, plus CSV, for each dataset. The original 11-location dataset is also kept separately.

In [ ]:
for package in ['xlwt', 'openpyxl']:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', package], check=True)
import xlwt

def export_dataset(frame, name):
    frame.to_csv(output_dir / f'{name}.csv', index=False)
    frame.to_excel(output_dir / f'{name}.xlsx', index=False, engine='openpyxl')
    workbook = xlwt.Workbook()
    sheet = workbook.add_sheet('Data')
    heading = xlwt.easyxf('font: bold on; pattern: pattern solid, fore_colour ice_blue;')
    sheet.set_panes_frozen(True)
    sheet.set_horz_split_pos(1)
    for column, label in enumerate(frame.columns):
        sheet.write(0, column, label, heading)
        sheet.col(column).width = 6000 if label != 'Address' else 15000
    for row_number, row in enumerate(frame.itertuples(index=False, name=None), start=1):
        for column, value in enumerate(row):
            if pd.isna(value):
                continue
            if hasattr(value, 'item'):
                value = value.item()
            sheet.write(row_number, column, value)
    workbook.save(str(output_dir / f'{name}.xls'))

exports = {'attractions_weather': Dataset, 'Attraction_EU': Attraction_EU, 'Iran': Iran}
for name, frame in exports.items():
    export_dataset(frame, name)
print('CSV, XLS and XLSX files saved for all three datasets.')

## 16. Verify and download all results

The completion message appears only after the database and exported files pass the checks below. These are current weather observations, not a forecast or historical dataset. Source coordinates are saved with the results.

In [ ]:
with sqlite3.connect(db_path) as connection:
    sql_attractions = pd.read_sql_query('SELECT * FROM attractions_weather', connection)
    sql_iran = pd.read_sql_query('SELECT * FROM Iran', connection)
assert len(sql_attractions) == len(Attraction_EU)
assert len(sql_iran) == 5
assert not sql_attractions.duplicated(['Longitude', 'Latitude']).any()
for name, frame in exports.items():
    assert len(pd.read_csv(output_dir / f'{name}.csv')) == len(frame)
    assert len(pd.read_excel(output_dir / f'{name}.xlsx')) == len(frame)
    assert (output_dir / f'{name}.xls').read_bytes()[:8] == bytes.fromhex('D0CF11E0A1B11AE1')
manifest = {
    'collected_at_utc': datetime.now(timezone.utc).isoformat(),
    'project': 'Weather API Data Collection',
    'country_selected': 'Iran', 'cities': iran_cities,
    'row_counts': {name: len(frame) for name, frame in exports.items()},
    'units': {'temperature': 'Celsius', 'wind': 'm/s', 'rain': 'mm/hour'},
    'missing_rain': 'Not supplied by provider; left blank',
    'County_column': 'OpenWeather locality name',
    'providers': ['LocationIQ / OpenStreetMap', 'OpenWeather'],
    'attribution': ['https://locationiq.com/attribution', 'https://www.openstreetmap.org/copyright']
}
(output_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive_path = Path(shutil.make_archive(str(output_dir), 'zip', root_dir=output_dir))
assert zipfile.is_zipfile(archive_path)
summary = pd.DataFrame([
    {'Dataset': name, 'Rows': len(frame), 'CSV': 'PASS', 'XLS': 'PASS', 'XLSX': 'PASS'}
    for name, frame in exports.items()
])
display(summary)
print('SQLite tables: attractions_weather and Iran - PASS')
print('DATA COLLECTION COMPLETE')
print('Download file:', archive_path.name)
from IPython.display import FileLink
display(FileLink(str(archive_path)))

### Keep your output files

Run the following cell to download the ZIP. If browser downloads are blocked, open Colab's Files panel, locate the printed output folder and download the ZIP from its menu. Colab runtime storage is temporary.

In [ ]:
from google.colab import files
from IPython.display import HTML, display
import base64

zip_data = base64.b64encode(archive_path.read_bytes()).decode('ascii')
display(HTML(f'<a download="{archive_path.name}" href="data:application/zip;base64,{zip_data}">Download results ZIP</a>'))
print('A copy of the ZIP is embedded in this saved notebook output.')
files.download(str(archive_path))